In [1]:
import json
import uuid
from datetime import datetime
from typing import TypedDict, Optional, Literal
 
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from neo4j import GraphDatabase
 
from memory_retriever import MemoryRetriever

In [ ]:
NEO4J_URI      = "bolt://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "xxx"
OLLAMA_MODEL   = "llama3"

In [3]:
INTENTS = Literal[
    "transactional",   # order history, what did I buy
    "temporal",        # when, delivery times, trends
    "behavioral",      # how I pay, what categories I buy
    "emotional",       # reviews, satisfaction, complaints
    "semantic",        # what is X, explain a concept
    "full",            # general / unknown → fetch everything
]

In [4]:
# ─────────────────────────────────────────────
# AGENT STATE
# ─────────────────────────────────────────────
class AgentState(TypedDict):
    # Input
    user_message       : str
    session_id         : str
 
    # Extracted
    customer_id        : Optional[str]
    intent             : Optional[str]
 
    # Memory context fetched from Neo4j
    memory_context     : Optional[dict]
 
    # Output
    response           : Optional[str]
 
    # Episodic tracking
    topics_discussed   : list[str]

In [5]:
# ─────────────────────────────────────────────
# INIT — LLM + Retriever
# ─────────────────────────────────────────────
llm       = ChatOllama(model=OLLAMA_MODEL, temperature=0)
retriever = MemoryRetriever(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
driver    = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

✅ MemoryRetriever connected to Neo4j


In [6]:
# ─────────────────────────────────────────────
# NODE 1 — ENTITY EXTRACTION
# Extract customer_id and intent from user message
# ─────────────────────────────────────────────
def extract_entities(state: AgentState) -> AgentState:
    print(f"\n[Node 1] Extracting entities from: '{state['user_message']}'")
 
    prompt = f"""
You are an entity extractor for an e-commerce assistant.
Extract information from the user message below.
 
User message: "{state['user_message']}"
 
Return ONLY a valid JSON object with these fields:
{{
  "customer_id": "<customer unique_id if mentioned, else null>",
  "intent": "<one of: transactional, temporal, behavioral, emotional, semantic, full>"
}}
 
Intent classification rules:
- transactional : questions about orders, purchases, products bought, order status
- temporal      : questions about when, delivery time, late orders, trends over time
- behavioral    : questions about payment methods, spending habits, categories
- emotional     : questions about reviews, satisfaction, complaints, sentiment
- semantic      : questions about what something means, explain a concept, what is X
- full          : general questions, greetings, or anything not fitting above
 
Return only the JSON. No explanation.
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()
 
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()
 
    try:
        extracted = json.loads(raw)
        customer_id = extracted.get("customer_id") or state.get("customer_id")
        intent      = extracted.get("intent", "full")
    except Exception as e:
        print(f"  ⚠️ Extraction parse error: {e}. Defaulting to full.")
        customer_id = state.get("customer_id")
        intent      = "full"
 
    print(f"  → customer_id={customer_id}, intent={intent}")
    return {**state, "customer_id": customer_id, "intent": intent}
 
 
# ─────────────────────────────────────────────
# ROUTER — conditional edge after extraction
# Routes to the right memory retrieval node
# ─────────────────────────────────────────────
def route_intent(state: AgentState) -> str:
    intent = state.get("intent", "full")
    print(f"\n[Router] Routing to: {intent}")
    return intent
 
 
# ─────────────────────────────────────────────
# NODE 3a — TRANSACTIONAL MEMORY RETRIEVAL
# ─────────────────────────────────────────────
def retrieve_transactional(state: AgentState) -> AgentState:
    print("[Node 3a] Fetching transactional memory...")
    cid = state.get("customer_id")
    if not cid:
        return {**state, "memory_context": {"error": "No customer_id found"}}
 
    context = {
        "memory_type"  : "transactional",
        "order_history": retriever.get_order_history(cid),
        "profile"      : retriever.get_customer_preferences(cid),
    }
 
    # Also fetch items for most recent order
    if context["order_history"]:
        latest_order_id = context["order_history"][0]["order_id"]
        context["latest_order_items"] = retriever.get_order_items(latest_order_id)
 
    print(f"  → {len(context['order_history'])} orders fetched")
    return {**state, "memory_context": context, "topics_discussed": ["order_history"]}
 
 
# ─────────────────────────────────────────────
# NODE 3b — TEMPORAL MEMORY RETRIEVAL
# ─────────────────────────────────────────────
def retrieve_temporal(state: AgentState) -> AgentState:
    print("[Node 3b] Fetching temporal memory...")
    cid = state.get("customer_id")
    if not cid:
        return {**state, "memory_context": {"error": "No customer_id found"}}
 
    context = {
        "memory_type"    : "temporal",
        "purchase_timeline": retriever.get_purchase_timeline(cid),
        "late_deliveries": retriever.get_late_deliveries(cid),
    }
    print(f"  → timeline={len(context['purchase_timeline'])} months, "
          f"late={len(context['late_deliveries'])} orders")
    return {**state, "memory_context": context, "topics_discussed": ["temporal_patterns"]}
 
 
# ─────────────────────────────────────────────
# NODE 3c — BEHAVIORAL MEMORY RETRIEVAL
# ─────────────────────────────────────────────
def retrieve_behavioral(state: AgentState) -> AgentState:
    print("[Node 3c] Fetching behavioral memory...")
    cid = state.get("customer_id")
    if not cid:
        return {**state, "memory_context": {"error": "No customer_id found"}}
 
    context = {
        "memory_type"      : "behavioral",
        "payment_behavior" : retriever.get_payment_behavior(cid),
        "category_behavior": retriever.get_category_behavior(cid),
        "preferences"      : retriever.get_customer_preferences(cid),
    }
    print(f"  → payment types={len(context['payment_behavior'])}, "
          f"categories={len(context['category_behavior'])}")
    return {**state, "memory_context": context, "topics_discussed": ["behavior", "preferences"]}
 
 
# ─────────────────────────────────────────────
# NODE 3d — EMOTIONAL MEMORY RETRIEVAL
# ─────────────────────────────────────────────
def retrieve_emotional(state: AgentState) -> AgentState:
    print("[Node 3d] Fetching emotional memory...")
    cid = state.get("customer_id")
    if not cid:
        return {**state, "memory_context": {"error": "No customer_id found"}}
 
    context = {
        "memory_type"      : "emotional",
        "review_history"   : retriever.get_review_history(cid),
        "emotional_summary": retriever.get_emotional_summary(cid),
    }
    print(f"  → {context['emotional_summary']}")
    return {**state, "memory_context": context, "topics_discussed": ["reviews", "sentiment"]}
 
 
# ─────────────────────────────────────────────
# NODE 3e — SEMANTIC MEMORY RETRIEVAL
# ─────────────────────────────────────────────
def retrieve_semantic(state: AgentState) -> AgentState:
    print("[Node 3e] Fetching semantic memory...")
 
    # Extract concept from message for semantic lookup
    concept_prompt = f"""
Extract the main concept or entity being asked about in this message.
Return ONLY the concept as a single word or short phrase. No explanation.
Message: "{state['user_message']}"
"""
    concept_response = llm.invoke([HumanMessage(content=concept_prompt)])
    concept = concept_response.content.strip().lower()
 
    context = {
        "memory_type"  : "semantic",
        "concept"      : concept,
        "facts"        : retriever.get_semantic_facts(concept),
        "all_facts"    : retriever.get_all_semantic_facts(),
    }
    print(f"  → concept='{concept}', facts={len(context['facts'])}")
    return {**state, "memory_context": context, "topics_discussed": [f"semantic:{concept}"]}
 
 
# ─────────────────────────────────────────────
# NODE 3f — FULL MEMORY RETRIEVAL (fallback)
# ─────────────────────────────────────────────
def retrieve_full(state: AgentState) -> AgentState:
    print("[Node 3f] Fetching full memory context...")
    cid = state.get("customer_id")
    if not cid:
        return {**state, "memory_context": {
            "memory_type": "full",
            "note": "No customer_id in message. Responding generally."
        }}
 
    context = {
        "memory_type": "full",
        **retriever.get_full_customer_context(cid)
    }
    print(f"  → Full context fetched for {cid}")
    return {**state, "memory_context": context, "topics_discussed": ["full_context"]}
 
 
# ─────────────────────────────────────────────
# NODE 4 — REASONING
# LLM synthesizes answer using memory context
# ─────────────────────────────────────────────
def reason(state: AgentState) -> AgentState:
    print("\n[Node 4] Reasoning with LLaMA3...")
 
    context = state.get("memory_context", {})
    memory_type = context.get("memory_type", "unknown")
 
    system_prompt = """You are a helpful e-commerce assistant with access to a customer's 
memory graph. You have deep knowledge of their order history, preferences, behavior, 
and emotional sentiment. Use the provided memory context to give precise, personalized answers.
Be concise and direct. If memory context is empty or missing, say so honestly."""
 
    user_prompt = f"""
Customer question: "{state['user_message']}"
 
Memory context ({memory_type}):
{json.dumps(context, indent=2, default=str)}
 
Answer the customer's question using only the memory context above.
Be specific — use actual values from the context (order IDs, categories, scores etc).
"""
 
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
 
    answer = response.content.strip()
    print(f"  → Response generated ({len(answer)} chars)")
    return {**state, "response": answer}
 
 
# ─────────────────────────────────────────────
# NODE 5 — EPISODIC MEMORY WRITE
# Save this conversation turn to Neo4j
# ─────────────────────────────────────────────
def write_episode(state: AgentState) -> AgentState:
    print("\n[Node 5] Writing episodic memory to Neo4j...")
 
    session_id  = state.get("session_id", str(uuid.uuid4()))
    customer_id = state.get("customer_id")
    topics      = state.get("topics_discussed", [])
    intent      = state.get("intent", "unknown")
    timestamp   = datetime.now().isoformat()
 
    with driver.session() as neo_session:
        neo_session.run("""
            MERGE (s:Session {session_id: $session_id})
            SET s.timestamp = $timestamp,
                s.intent    = $intent
 
            WITH s
            UNWIND $topics AS topic
            MERGE (t:Topic {name: topic})
            MERGE (s)-[:DISCUSSED_TOPIC]->(t)
 
            WITH s
            FOREACH (_ IN CASE WHEN $customer_id IS NOT NULL THEN [1] ELSE [] END |
                MERGE (c:Customer {unique_id: $customer_id})
                MERGE (s)-[:INVOLVED_CUSTOMER]->(c)
            )
        """,
            session_id  = session_id,
            timestamp   = timestamp,
            intent      = intent,
            topics      = topics,
            customer_id = customer_id,
        )
 
    print(f"  → Episode saved: session={session_id}, topics={topics}")
    return {**state, "session_id": session_id}
 
 

In [7]:
def build_graph() -> StateGraph:
    graph = StateGraph(AgentState)
 
    # Add all nodes
    graph.add_node("extract_entities",       extract_entities)
    graph.add_node("retrieve_transactional", retrieve_transactional)
    graph.add_node("retrieve_temporal",      retrieve_temporal)
    graph.add_node("retrieve_behavioral",    retrieve_behavioral)
    graph.add_node("retrieve_emotional",     retrieve_emotional)
    graph.add_node("retrieve_semantic",      retrieve_semantic)
    graph.add_node("retrieve_full",          retrieve_full)
    graph.add_node("reason",                 reason)
    graph.add_node("write_episode",          write_episode)
 
    # Entry point
    graph.set_entry_point("extract_entities")
 
    # Conditional routing after extraction
    graph.add_conditional_edges(
        "extract_entities",
        route_intent,
        {
            "transactional" : "retrieve_transactional",
            "temporal"      : "retrieve_temporal",
            "behavioral"    : "retrieve_behavioral",
            "emotional"     : "retrieve_emotional",
            "semantic"      : "retrieve_semantic",
            "full"          : "retrieve_full",
        }
    )
 
    # All retrieval nodes → reason
    for node in [
        "retrieve_transactional",
        "retrieve_temporal",
        "retrieve_behavioral",
        "retrieve_emotional",
        "retrieve_semantic",
        "retrieve_full",
    ]:
        graph.add_edge(node, "reason")
 
    # reason → write_episode → END
    graph.add_edge("reason",        "write_episode")
    graph.add_edge("write_episode", END)
 
    return graph.compile()

In [8]:
# ─────────────────────────────────────────────
# CONVERSATION RUNNER
# ─────────────────────────────────────────────
class MemoryAgent:
    def __init__(self):
        self.graph      = build_graph()
        self.session_id = str(uuid.uuid4())
        self.customer_id = None
        print(f"✅ MemoryAgent ready. Session: {self.session_id}")
 
    def chat(self, message: str) -> str:
        state = AgentState(
            user_message     = message,
            session_id       = self.session_id,
            customer_id      = self.customer_id,
            intent           = None,
            memory_context   = None,
            response         = None,
            topics_discussed = [],
        )
 
        result = self.graph.invoke(state)
 
        # Persist customer_id across turns
        if result.get("customer_id"):
            self.customer_id = result["customer_id"]
 
        return result.get("response", "I could not generate a response.")
 
 

In [9]:
TEST_CUSTOMER = "b5d6fa3d2213927296ac893f14f4461c"

In [10]:
agent = MemoryAgent()

✅ MemoryAgent ready. Session: 38b0889c-2beb-47af-a35c-825878d6432b


In [11]:
test_conversations = [
        # Transactional
        (f"What orders has customer {TEST_CUSTOMER} placed?",
         "transactional"),
 
        # Temporal
        (f"Were any orders for customer {TEST_CUSTOMER} delivered late?",
         "temporal"),
 
        # Behavioral
        (f"How does customer {TEST_CUSTOMER} usually pay?",
         "behavioral"),
 
        # Emotional
        (f"What is the review sentiment for customer {TEST_CUSTOMER}?",
         "emotional"),
 
        # Semantic
        ("What is boleto as a payment method?",
         "semantic"),
 
        # Full / general
        (f"Give me a full profile of customer {TEST_CUSTOMER}",
         "full"),
    ]
 

In [12]:
print("\n" + "="*60)
print("MEMORY AGENT — TEST CONVERSATIONS")
print("="*60)


MEMORY AGENT — TEST CONVERSATIONS


In [14]:
for message, expected_intent in test_conversations:
        print(f"\n{'─'*60}")
        print(f"🧑 User    : {message}")
        print(f"📌 Expected intent: {expected_intent}")
        response = agent.chat(message)
        print(f"🤖 Agent   : {response}")
 
print("\n" + "="*60)
print("✅ All test conversations complete.")
print("Check Neo4j for episodic memory: MATCH (s:Session) RETURN s")
 


────────────────────────────────────────────────────────────
🧑 User    : What orders has customer b5d6fa3d2213927296ac893f14f4461c placed?
📌 Expected intent: transactional

[Node 1] Extracting entities from: 'What orders has customer b5d6fa3d2213927296ac893f14f4461c placed?'
  → customer_id=b5d6fa3d2213927296ac893f14f4461c, intent=transactional

[Router] Routing to: transactional
[Node 3a] Fetching transactional memory...
  → 1 orders fetched

[Node 4] Reasoning with LLaMA3...
  → Response generated (395 chars)

[Node 5] Writing episodic memory to Neo4j...
  → Episode saved: session=38b0889c-2beb-47af-a35c-825878d6432b, topics=['order_history']
🤖 Agent   : According to the provided memory context, Customer b5d6fa3d2213927296ac893f14f4461c has placed one order:

* Order ID: afffe744c625af23449137b8f832e630
* Status: delivered
* Purchased at: 2018-02-25 19:13:33
* Year: 2018
* Month: 2
* Item count: 3

This order contains three items from the "pet_shop" category, all sold by seller with